## 02 — The Comparison

This is the final notebook.

We put both approaches side by side: the system we built across seven modules and the one command that replaces most of it. The goal is not to show that `tippecanoe` is better — it obviously is for production use — but to show exactly **what work it is doing**, because we built every one of those pieces ourselves.

## System Comparison — Architecture

```
OUR SYSTEM                                  TIPPECANOE + TILE CLIENT
─────────────────────────────────────       ──────────────────────────────────────
ne_10m_railroads.geojson (40 MB)            ne_10m_railroads.geojson (40 MB)
       │                                           │
       ▼                                           ▼
Module 02: simplify at 4 epsilons           tippecanoe (one command, ~30 seconds)
  → 4 GeoJSON output files                        │
       │                                           ▼
       ▼                                    railroads.pmtiles (~3 MB)
Module 04: build 4 GridIndex objects               │
  (startup: ~5 seconds)                            ▼
       │                               tile client (browser or localtileserver)
       ▼                                 fetches only visible tiles on demand
Module 05: get_lod(zoom) selects index
       │
       ▼
Module 03: bbox cull within selected index
       │
       ▼
GeoJSON layer (re-sent every pan/zoom)
```

## System Comparison — Numbers

In [ ]:
from pathlib import Path

lod_dir = Path("../../data/lod")
raw     = Path("../../data/ne_10m_railroads.geojson")
pmtiles = Path("../../data/railroads.pmtiles")

lod_files = [
    "railroads_coarse.geojson",
    "railroads_medium.geojson",
    "railroads_fine.geojson",
    "railroads_extra_fine.geojson",
]

our_total_mb = sum((lod_dir / f).stat().st_size for f in lod_files) / 1_000_000
raw_mb       = raw.stat().st_size / 1_000_000
pm_mb        = pmtiles.stat().st_size / 1_000_000 if pmtiles.exists() else None

print("Storage comparison:")
print(f"  Raw GeoJSON:               {raw_mb:.1f} MB")
print(f"  Our 4 LOD files (total):   {our_total_mb:.1f} MB")
if pm_mb:
    print(f"  tippecanoe PMTiles:        {pm_mb:.1f} MB")
else:
    print("  tippecanoe PMTiles:        (run Notebook 01 first)")

print()
print("Runtime comparison:")
print(f"  Our startup (load + index): ~5–10s")
print(f"  Our per-query time:         ~1–5ms")
print(f"  Tile client startup:        ~0s (lazy fetch)")
print(f"  Tile fetch (per tile):      ~10–50ms over network")
print(f"  Tile fetch (local):         <1ms")

## What Tippecanoe Automated — Specifically

Now name each thing precisely — because you built it:

**1. Multi-resolution simplification (our Module 02)**
tippecanoe applies a tolerance appropriate for each zoom level automatically. You do not choose 4 epsilons — you choose one `--simplification` value and it scales it per zoom.

**2. Spatial bucketing (our Module 04)**
Tiles are the index. There is no separate grid index to build — the tile `(z, x, y)` address is the bucket. Features are pre-assigned to tiles at generation time.

**3. Viewport culling (our Module 03)**
The client requests only the tiles its viewport covers. No intersection test — irrelevant tiles are never fetched.

**4. LOD switching (our Module 05)**
The tile URL includes the zoom level (`/{z}/`). The client naturally requests tiles at the right zoom. No decision function needed.

**5. Binary encoding**
We never built this — our GeoJSON is text. Tippecanoe outputs MVT binary with integer coordinates. ~5× smaller and faster to parse.

**6. Streaming / lazy loading**
We never built this either. Tiles are fetched on demand; unused regions are never touched.

## What Tippecanoe Does NOT Do

Tippecanoe is a **preprocessing pipeline** — it generates static files. It does not:

- Serve tiles dynamically (you still need a static host, CDN, or `localtileserver`)
- Filter features at query time based on user input
- Handle real-time data updates
- Decide which features to show based on screen density or user preferences at render time

For live, user-driven filtering (e.g., "show only electrified railways"), a tile system either:
- Pre-generates multiple tile sets (one per filter combination)
- Sends all attributes in the tile and lets the client filter at render time (style expressions)
- Uses a dynamic tile server that queries a database per request

None of these are trivial. Our handbuilt system could add real-time filtering in ten lines.

## The Quote That Started This

> *"Build the smallest version that teaches the idea. Borrow the version that survives the real world."*

You built the smallest version. You know:
- What Douglas-Peucker does and why epsilon matters
- Why spatial indexes exist and what they trade off
- Why bounding box culling is O(n) without an index
- Why the tile coordinate scheme is itself a spatial index
- Why binary encoding is worth the complexity

When you use `tippecanoe` from now on, you can read its flags without guessing. When it produces unexpected output, you can reason about why. When a colleague says "we should just use vector tiles" without understanding the tradeoffs, you can ask the right questions.

That is not the same as having typed `tippecanoe` once.

## Exercise A

Write a one-page (or ~15 bullet point) technical comparison of the two systems. Cover:
- Build time
- Storage footprint
- Startup cost for the end user
- Per-request data transfer
- Support for real-time filtering
- Complexity to maintain
- What you would use for a class project vs. a production application

Write it in the cell below as markdown.

### Handbuilt LOD Viewer vs. Tippecanoe + Vector Tiles

**Build time**
- Our system: Module 02 pipeline runs Douglas-Peucker across 25,413 features at 4 epsilons. Takes minutes on first run. Output is committed to disk and reused.
- Tippecanoe: one command, ~30 seconds to generate the full zoom 1–14 pyramid.

**Storage footprint**
- Our system: 4 GeoJSON files totaling ~41 MB (text encoding, full-world coverage at each level).
- Tippecanoe PMTiles: typically ~3–5 MB for this dataset, binary MVT encoding, only non-empty tiles stored.

**Startup cost for the end user**
- Our system: ~5–10 seconds every session to load all 4 files and build 4 grid indexes before the map is usable.
- Tile client: ~0 seconds, tiles are fetched lazily as the user navigates. The first viewport loads in under a second.

**Per-request data transfer**
- Our system: up to 19 MB (the full extra_fine file) regardless of where the user is looking.
- Tiles: 10–200 KB per tile. A typical viewport at zoom 12 fetches 12–20 tiles (1–2 MB total), and only what is visible.

**Support for real-time filtering**
- Our system: trivial to add. A filter function in the `update()` handler can show/hide features by any property without rebuilding anything.
- Tile client: filtering must be done at generation time (separate tile sets per filter) or at render time via Mapbox GL style expressions on attributes embedded in the tile. Adding a new filter dimension can require re-running tippecanoe.

**Complexity to maintain**
- Our system: ~200 lines of readable Python. Every component is visible and modifiable. A developer new to the codebase can trace the full data path in one notebook.
- Tippecanoe pipeline: one command with flags, but the internals are a compiled C++ binary. Debugging unexpected output (missing features, wrong simplification) requires knowledge of how tippecanoe makes its decisions internally.

**What I would use for a class project**
- Our handbuilt system. It runs in a Jupyter notebook with no installation beyond standard Python packages, produces visible output at every step, and every line maps to a concept taught in the course. There is nothing opaque.

**What I would use for a production application**
- Tippecanoe + PMTiles on a CDN. The storage savings (41 MB → ~4 MB), zero startup cost, on-demand tile streaming, and browser-native rendering via MapLibre GL would make the handbuilt system impractical at any scale beyond a single notebook session.

## Exercise B

Look up the `--attribute-filter` and `--include` flags in the tippecanoe documentation.

Could you use these to produce a tile set that only includes electrified railroads (`electric` property)? Write the command you would use, and explain what the resulting tile set would and would not be able to show.

In [ ]:
# The railroad fine LOD has 7,184 features with electric=1 (electrified),
# 1,879 with electric=2 (partial/unknown), and 16,350 with electric=0 (not electrified).
#
# To produce a tile set containing only electrified lines, use --feature-filter
# with a Mapbox GL filter expression:
#
#   tippecanoe \
#     --output=railroads_electric.pmtiles \
#     --force \
#     --minimum-zoom=1 \
#     --maximum-zoom=14 \
#     --simplification=10 \
#     --drop-densest-as-needed \
#     --layer=railroads \
#     --feature-filter='["==", ["get", "electric"], 1]' \
#     ne_10m_railroads.geojson
#
# The --feature-filter flag accepts a Mapbox GL expression. ["==", ["get", "electric"], 1]
# keeps only features where the "electric" property equals 1. This is applied at
# tile generation time, so no electric=0 features ever enter the tile files.
#
# What the resulting tile set CAN show:
#   - Global distribution of electrified main-line rail networks
#   - High-voltage rail corridors (most high-speed lines are electric)
#   - Density comparisons between regions (Western Europe dense, US sparse)
#
# What it CANNOT show:
#   - The full railroad network for context, non-electric lines are absent entirely.
#     A user viewing the tile set would see large gaps in the US and developing regions
#     that have railroads but not electrified ones, which could be misread as no service.
#   - Features with electric=2 (partial electrification) are also excluded; including
#     them would require changing the filter to ["in", ["get", "electric"], ["literal", [1, 2]]].
#   - Any toggle between "all railroads" and "electrified only" that would require
#     generating two separate tile sets and switching between them in the client.

## Check Your Understanding

A classmate who skipped Modules 01–06 and came straight to this notebook could run `tippecanoe` and get a working tile set. They would see the flags but not know what they mean.

Name **three specific situations** where their lack of understanding would cost them — where they would make a wrong decision, miss a bug, or be unable to debug a problem — that you would be able to handle.

---

In [ ]:
# Three situations where skipping Modules 01-06 would cost a tippecanoe user:
#
# 1. Choosing --simplification without knowing what it means in practice.
#    A classmate who has not implemented Douglas-Peucker would not understand that
#    --simplification=10 means 10 tile pixels of tolerance, which is ~24 km at
#    zoom 2 but only ~24 m at zoom 12. They might raise it to 50 to shrink the
#    file and not realize they have smoothed out every small curve in city-level
#    rail geometry, making the map visibly wrong at high zoom. Someone who built
#    the epsilon tradeoff table in Module 02 would recognize the problem immediately.
#
# 2. Debugging a blank map at low zoom levels.
#    If --drop-densest-as-needed removes too many features at zoom 2-4 and the
#    map appears empty over large regions, a classmate who skipped the course would
#    have no framework for diagnosing why. They would not know that the coarse
#    scalerank filter is the right fix, explicitly keeping only scalerank <= 4
#    features ensures the globally important skeleton is never dropped. Someone
#    who built the LOD pipeline and explored the scalerank distribution in Module 00
#    would recognize the symptom and know to add --feature-filter or a preprocessing
#    step immediately.
#
# 3. Misreading slow viewport updates as a network problem.
#    A tile client that fetches many large tiles slowly could be caused by: tile
#    sizes that are too large (--maximum-zoom too high), too many features per tile
#    (no simplification or filtering), or genuinely slow network. A classmate with
#    no background in spatial indexing would not know which dimension to tune.
#    Someone who benchmarked grid index query times in Module 04 and measured
#    per-feature culling costs in Module 03 understands that the unit of work at
#    high zoom is the tile, and that the right fix is usually reducing tile
#    complexity, not improving network speed.

## End of the Data Manager Micro Lessons

You built:
- A simplification algorithm
- A multi-resolution data pipeline
- A spatial intersection test
- A grid-based spatial index
- A zoom-driven data selection system
- A working interactive map viewer
- An informed opinion about when to stop building and borrow instead

The railroad project is where you apply it.